# Exercise 06 — Positional Encodings in Practice

Self-attention on its own treats its input as a *set*: shuffle the tokens and the outputs are shuffled in the same way (you showed this in week 5). Information about order has to be **put in**. So far you have seen two ways of doing that: the **sinusoidal** encoding of the original Transformer (week 5) and the **learned** position embeddings of GPT-2 and your mini GPT (notebook 0). Both are *absolute*: a vector for "position 17" is added to the token embedding.

Today's language models mostly do something different. Llama, Mistral, Qwen (which you will fine-tune in week 7) and many others use **rotary position embeddings (RoPE)**. BLOOM and MPT use **ALiBi**. And a decoder with *no* positional encoding at all (**NoPE**) works surprisingly well. These methods encode *relative* position inside the attention instead of adding absolute positions to the input.

In this notebook you implement RoPE and ALiBi, plug all five methods into the same mini GPT, and compare them **empirically**:
- How well do they model language at the length they were trained on?
- What happens on sequences **longer than any seen in training**?
- Can they solve a task that needs **exact relative positions**?

| method | where | what | used in |
|---|---|---|---|
| none (NoPE) | — | nothing; only the causal mask carries order information | some layers of recent models |
| learned | input | a trained vector per position, added to the token embedding | BERT, GPT-2, your mini GPT |
| sinusoidal | input | a fixed sine/cosine vector per position, added to the token embedding | the original Transformer |
| RoPE | attention | queries and keys are *rotated* by an angle proportional to their position | Llama, Mistral, Qwen, Gemma, … |
| ALiBi | attention | a penalty proportional to the distance is added to the attention scores | BLOOM, MPT |

## What you will do
1. **RoPE**: rotations, and why the attention score then depends only on the relative position.
2. **ALiBi**: distance penalties, and the attention windows they create.
3. **One model, five encodings**: a mini GPT with a switch for the positional encoding.
4. **Language modelling**: train all five on Shakespeare.
5. **Length extrapolation**: evaluate on sequences four times longer than in training.
6. **A task that needs exact positions**: "what was the token five steps ago?"
7. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Compute.** Everything runs on a laptop CPU. The models are deliberately tiny, so that all ten training runs together take roughly 5–10 minutes. The *differences* between the methods are what matters here, not the absolute quality of the models. The notebook reuses Tiny Shakespeare (`input.txt`) from notebook 0, and downloads it if it is not there.

In [ ]:
%matplotlib inline
import math
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

POSITION_METHODS = ["none", "learned", "sinusoidal", "rope", "alibi"]

## 1. RoPE: rotary position embeddings

[RoPE](https://arxiv.org/abs/2104.09864) (Su et al., 2021) does not add anything to the token embeddings. Instead, inside every attention layer, it **rotates** each query and each key by an angle that grows with the token's position.

Take a 2-dimensional query $q$ at position $m$ and a key $k$ at position $n$. RoPE rotates $q$ by the angle $m\theta$ and $k$ by $n\theta$. Since the dot product of two vectors only depends on their lengths and the angle *between* them,

$$\langle R_{m\theta}\, q,\; R_{n\theta}\, k \rangle = \langle q,\; R_{(n-m)\theta}\, k\rangle ,$$

the attention score depends on the positions only through their **difference** $n - m$. Absolute positions go in, but the model only ever sees relative ones.

A head with `head_dim` dimensions is split into `head_dim / 2` pairs, and every pair is rotated with its own frequency $\theta_i = 10000^{-2i/\text{head\_dim}}$, the same frequencies as in the sinusoidal encoding. Fast-rotating pairs resolve small distances, slow ones large distances. As in Llama, we pair dimension $i$ with dimension $i + \text{head\_dim}/2$. The rotation of all pairs at once can then be written as

$$\text{RoPE}(x) = x \odot \cos(\text{angles}) + \text{rotate\_half}(x) \odot \sin(\text{angles}),$$

where `rotate_half` turns $(x_1, x_2)$ (the two halves of $x$) into $(-x_2, x_1)$, and every angle appears twice, once for each half.

**1.1 Complete `rope_cos_sin` and `apply_rope`.**

Hint: `torch.outer(positions, inv_freq)` gives the matrix of all products position × frequency.

In [ ]:
def rope_cos_sin(positions, head_dim, base=10000.0):
    # positions: (T,) float tensor -> cos and sin of the rotation angles, each of shape (T, head_dim)
    inv_freq = 1.0 / base ** (torch.arange(0, head_dim, 2, device=positions.device).float() / head_dim)  # (head_dim / 2,)
    angles = ...  # TODO: the angle of every (position, frequency) pair, (T, head_dim / 2)
    angles = torch.cat([angles, angles], dim=-1)  # each angle is used for both halves, (T, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # (x1, x2) -> (-x2, x1), where x1 and x2 are the two halves of the last dimension
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (..., T, head_dim); cos, sin: (T, head_dim) -> x rotated, same shape as x
    return ...  # TODO

In [ ]:
# ✅ Check your RoPE
torch.manual_seed(0)
q, k = torch.randn(32), torch.randn(32)


def rope_score(q, k, m, n, head_dim=32):
    # The dot product of q rotated to position m and k rotated to position n
    cos, sin = rope_cos_sin(torch.tensor([float(m), float(n)]), head_dim)
    return (apply_rope(q, cos[0], sin[0]) * apply_rope(k, cos[1], sin[1])).sum()


cos, sin = rope_cos_sin(torch.arange(10).float(), 32)
assert cos.shape == sin.shape == (10, 32), "cos and sin should have shape (T, head_dim)"
assert torch.allclose(apply_rope(q, cos[0], sin[0]), q), "position 0 means an angle of 0: nothing is rotated"
assert torch.isclose(apply_rope(q, cos[7], sin[7]).norm(), q.norm()), "a rotation does not change the length of a vector"
assert torch.isclose(rope_score(q, k, 3, 1), rope_score(q, k, 103, 101), atol=1e-3), "the score should depend only on the relative position"
assert not torch.isclose(rope_score(q, k, 3, 1), rope_score(q, k, 3, 2), atol=1e-3), "a different relative position should give a different score"
print("Looks good ✅")

**1.2 Run the cell below.** It plots the RoPE score of a vector with itself, $\langle R_{m\theta}\, q, R_{n\theta}\, q\rangle$, as a function of the distance $n - m$ (scaled so that distance 0 gives 1), for a vector of ones and averaged over many random vectors.

In [ ]:
head_dim = 32
distances = torch.arange(0, 1025).float()
cos, sin = rope_cos_sin(distances, head_dim)

ones = torch.ones(head_dim)
score_ones = (apply_rope(ones, cos, sin) * ones).sum(-1) / head_dim

random_q = torch.randn(200, 1, head_dim)
score_random = (apply_rope(random_q, cos, sin) * random_q).sum(-1).mean(0) / head_dim

plt.figure(figsize=(9, 3))
plt.plot(distances, score_ones, label="q = k = ones")
plt.plot(distances, score_random, label="q = k, random (average)")
plt.axhline(0, color="gray", linewidth=0.5)
plt.xscale("symlog", linthresh=10)
plt.xlabel("relative distance n - m")
plt.ylabel("score")
plt.legend()
plt.show()

**1.3 What does the plot show?** What happens to the score between two *identical* vectors as they move further apart? Why could that be a useful property for a language model? And why do the slowest-rotating pairs matter for sequences that are much longer than those seen in training?

---

*Your answer here:*

---

## 2. ALiBi: attention with linear biases

[ALiBi](https://arxiv.org/abs/2108.12409) (Press et al., 2022) is even simpler. It adds nothing to the embeddings and changes no vectors. It adds a **penalty proportional to the distance** to the attention scores, before the softmax:

$$\text{score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d}} - m_h \cdot (i - j)$$

Every head $h$ has its own fixed slope $m_h$. For $H$ heads, the slopes are the geometric sequence $m_h = 2^{-8h/H}$ for $h = 1, \dots, H$: with 4 heads, $\tfrac{1}{4}, \tfrac{1}{16}, \tfrac{1}{64}, \tfrac{1}{256}$. Steep heads look only at the last few tokens, flat heads far back. There are no parameters to learn, and the penalty is defined for any distance, so ALiBi was designed to **extrapolate**: "train short, test long" is the title of the paper.

**2.1 Complete `alibi_slopes` and `alibi_bias`.**

Hints:
- `torch.arange(1, n_head + 1)` gives $h = 1, \dots, H$.
- `i[:, None] - j[None, :]` with `i = j = torch.arange(T)` gives the matrix of distances $i - j$. (Entries above the diagonal become positive; they are masked by the causal mask anyway.)

In [ ]:
def alibi_slopes(n_head):
    # The slope m_h of every head, (n_head,)
    return ...  # TODO


def alibi_bias(slopes, T):
    # The bias -m_h * (i - j) for every head h, query position i and key position j, (n_head, T, T)
    positions = torch.arange(T, device=slopes.device)
    distance = ...  # TODO: (T, T), distance[i, j] = i - j
    return ...      # TODO: (n_head, T, T)

In [ ]:
# ✅ Check your ALiBi
slopes = alibi_slopes(4)
assert torch.allclose(slopes, torch.tensor([1 / 4, 1 / 16, 1 / 64, 1 / 256])), f"with 4 heads the slopes are 1/4, 1/16, 1/64, 1/256, got {slopes}"
bias = alibi_bias(slopes, 6)
assert bias.shape == (4, 6, 6), "the bias should have shape (n_head, T, T)"
assert torch.all(bias.diagonal(dim1=1, dim2=2) == 0), "a token attending to itself gets no penalty"
assert math.isclose(bias[0, 5, 2].item(), -0.75), "head 0, query 5, key 2: distance 3, slope 1/4"
print("Looks good ✅")

**2.2 Run the cell below.** It shows the attention weights of the last position of a 256-token sequence if the queries and keys carried *no* information, so that the ALiBi bias alone decides: one curve per head.

In [ ]:
T = 256
weights = torch.softmax(alibi_bias(alibi_slopes(4), T)[:, -1, :], dim=-1)  # (n_head, T): the query at the last position

plt.figure(figsize=(9, 3))
for h in range(4):
    plt.plot(torch.arange(T - 1, -1, -1), weights[h], label=f"head {h}, slope 1/{int(1 / alibi_slopes(4)[h])}")
plt.xlabel("distance to the query (tokens back)")
plt.ylabel("attention weight")
plt.xscale("symlog", linthresh=1)
plt.legend()
plt.show()

for h in range(4):
    print(f"head {h}: penalty at distance 200 = {alibi_bias(alibi_slopes(4), 201)[h, 200, 0]:.1f}")

**2.3 What does ALiBi do to the attention, and what is the price?** Think about a question near the end of a long document whose answer is 1,000 tokens earlier.

---

*Your answer here:*

---

## 3. One model, five positional encodings

Below is the mini GPT from notebook 0, with a switch `pos` for the positional encoding. Three details differ from notebook 0:
- The causal mask is built in every forward pass for the actual length `T`, so the model accepts sequences of **any length** (except with learned positions, which exist only up to `block_size`).
- For `"sinusoidal"`, the encoding is precomputed for up to `max_len = 4096` positions and scaled to the size of the (initial) token embeddings, so that it does not drown out the tokens. `sinusoidal_encoding` is the function you wrote in week 5.
- The loss ignores targets with the value `-100` (we need that in Part 6).

**3.1 Complete the position-specific parts**: the two absolute encodings in `GPT.forward`, and RoPE and ALiBi in `CausalSelfAttention.forward`.

Hints:
- Learned positions: `self.pos_emb(torch.arange(T, device=idx.device))`, as in notebook 0.
- RoPE rotates the **queries and keys** of all heads (not the values). `q` has shape `(B, n_head, T, head_dim)`, and `cos` and `sin` of shape `(T, head_dim)` broadcast over the batch and the heads.
- `alibi_bias(self.slopes, T)` has shape `(n_head, T, T)` and broadcasts over the batch.

In [ ]:
def sinusoidal_encoding(max_len, d):
    # The sinusoidal encoding of the original Transformer (week 5), (max_len, d)
    position = torch.arange(max_len).float().unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe = torch.zeros(max_len, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, pos):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.pos = pos
        self.rope_scale = 1.0  # positions are multiplied by this factor before the rotation (used in Part 5)
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        if pos == "alibi":
            self.register_buffer("slopes", alibi_slopes(n_head))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))  # (B, n_head, T, head_dim)

        if self.pos == "rope":
            positions = torch.arange(T, device=x.device).float() * self.rope_scale
            cos, sin = rope_cos_sin(positions, self.head_dim)
            q = ...  # TODO
            k = ...  # TODO

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)  # (B, n_head, T, T)
        if self.pos == "alibi":
            scores = ...  # TODO: add the ALiBi bias

        causal = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        scores = scores.masked_fill(~causal, float("-inf"))
        out = torch.softmax(scores, dim=-1) @ v
        return self.proj(out.transpose(1, 2).reshape(B, T, C))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, pos):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, pos)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, pos, block_size, n_embd, n_head, n_layer, max_len=4096):
        super().__init__()
        assert pos in POSITION_METHODS, f"unknown positional encoding {pos!r}"
        self.pos = pos
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        if pos == "learned":
            self.pos_emb = nn.Embedding(block_size, n_embd)
        if pos == "sinusoidal":
            self.register_buffer("pe", sinusoidal_encoding(max_len, n_embd) * 0.02 * math.sqrt(2))  # scaled to std 0.02
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, pos) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.pos == "learned":
            if T > self.block_size:
                raise ValueError(f"learned position embeddings only exist for {self.block_size} positions, got {T}")
            x = ...  # TODO: add the learned position embeddings
        if self.pos == "sinusoidal":
            x = ...  # TODO: add the first T rows of the sinusoidal encoding
        logits = self.lm_head(self.ln_f(self.blocks(x)))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        return logits, loss

The check below tests every variant for causality and for whether it can tell positions apart. For the second test it uses a **one-layer** model and swaps the first two tokens: the output at position 2 must change, because the same three tokens now come in a different order.

In [ ]:
# ✅ Check your model
torch.manual_seed(0)
for pos in POSITION_METHODS:
    m = GPT(vocab_size=10, pos=pos, block_size=16, n_embd=32, n_head=4, n_layer=2)
    x, y = torch.randint(10, (2, 16)), torch.randint(10, (2, 16))
    with torch.no_grad():
        logits, loss = m(x, y)
        assert logits.shape == (2, 16, 10), f"{pos}: wrong logits shape {tuple(logits.shape)}"
        assert abs(loss.item() - math.log(10)) < 0.3, f"{pos}: an untrained model should have a loss close to log(10)"
        changed = x.clone()
        changed[:, 8:] = (changed[:, 8:] + 1) % 10
        assert torch.allclose(m(changed)[0][:, :8], logits[:, :8], atol=1e-5), f"{pos}: the model can see the future"

        one_layer = GPT(vocab_size=10, pos=pos, block_size=16, n_embd=32, n_head=4, n_layer=1)
        nn.init.normal_(one_layer.blocks[0].attn.qkv.weight, std=1.0)  # larger scores, so that small effects are visible
        abc, bac = torch.tensor([[1, 2, 3]]), torch.tensor([[2, 1, 3]])
        same = torch.allclose(one_layer(abc)[0][0, 2], one_layer(bac)[0][0, 2], atol=1e-5)
        if pos == "none":
            assert same, "without positional encoding, one layer cannot tell 'a b c' from 'b a c' at position 2"
        else:
            assert not same, f"{pos}: the model does not use the positions. Check your {pos} code"
    print(f"{pos:<11} ok")

try:
    GPT(vocab_size=10, pos="learned", block_size=16, n_embd=32, n_head=4, n_layer=1)(torch.zeros(1, 17, dtype=torch.long))
    raise AssertionError("learned positions should fail for sequences longer than block_size")
except ValueError:
    pass
print("Looks good ✅")

**3.2 Without any positional encoding, the one-layer model cannot tell *a b c* from *b a c* at position 2.** Yet decoders without positional encoding (NoPE) are known to work reasonably well, and you will train one in Part 4. Where can a causal decoder with two or more layers get information about positions from?

Hint: how many tokens can position 0 attend to, and how many can position 50?

---

*Your answer here:*

---

## 4. Language modelling on Shakespeare

First the everyday task: character-level language modelling on Tiny Shakespeare, as in notebook 0, but with a smaller model, so that five of them train in a few minutes. All five use the same data, the same seed and the same number of steps; only the positional encoding differs.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
with open(DATA_PATH, encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_text_batch(split, T, batch_size=32):
    # Random chunks of T characters (x) and the same chunks shifted by one (y)
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - T - 1, (batch_size,))
    x = torch.stack([d[i : i + T] for i in ix])
    y = torch.stack([d[i + 1 : i + T + 1] for i in ix])
    return x.to(device), y.to(device)


print(f"{vocab_size} characters, {len(train_data):,} for training")

`train` trains a model and records the training loss. It is the training loop from notebook 0 with a learning-rate warm-up and decay.

In [ ]:
# The model and the training run
BLOCK_SIZE = 64  # training length: the model never sees longer sequences during training
N_EMBD, N_HEAD, N_LAYER = 64, 4, 3
LM_STEPS = 1500


def make_model(pos, vocab_size, block_size, n_layer=N_LAYER):
    torch.manual_seed(1337)  # the same initial weights (where they exist) for every method
    return GPT(vocab_size, pos, block_size, N_EMBD, N_HEAD, n_layer).to(device)


def train(model, get_batch, steps, lr=2e-3):
    # Train with AdamW, a learning-rate warm-up and cosine decay; returns the training losses
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=steps, pct_start=0.1, cycle_momentum=False)
    model.train()
    losses = []
    for _ in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        schedule.step()
        losses.append(loss.item())
    model.eval()
    return losses


print(f"{sum(p.numel() for p in make_model('learned', vocab_size, BLOCK_SIZE).parameters()):,} parameters (with learned positions)")

**4.1 Train the five models.** This takes roughly 2–5 minutes on a CPU in total.

In [ ]:
lm_models, lm_losses = {}, {}
for pos in POSITION_METHODS:
    start = time.time()
    model = make_model(pos, vocab_size, BLOCK_SIZE)
    lm_losses[pos] = train(model, lambda: get_text_batch("train", BLOCK_SIZE), LM_STEPS)
    lm_models[pos] = model
    print(f"{pos:<11} final training loss {sum(lm_losses[pos][-100:]) / 100:.3f}   ({time.time() - start:.0f} s)")


def smooth(values, window=50):
    return [sum(values[max(0, i - window) : i + 1]) / len(values[max(0, i - window) : i + 1]) for i in range(len(values))]


plt.figure(figsize=(9, 3))
for pos, losses in lm_losses.items():
    plt.plot(smooth(losses), label=pos)
plt.ylim(1.4, 3.0)
plt.xlabel("training step")
plt.ylabel("training loss (smoothed)")
plt.legend()
plt.show()

To compare the models properly, and to look beyond the training length in Part 5, we need the validation loss **at every position** of a sequence.

**4.2 Complete `loss_per_position`**, which returns the validation loss at each of the `T` positions, averaged over `n_batches` batches of `T`-character chunks.

Hint: `F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), reduction="none")` returns one loss per position instead of the mean; reshape it back to `(batch_size, T)`.

In [ ]:
@torch.no_grad()
def loss_per_position(model, T, n_batches=20):
    # The validation loss at each position 0 .. T-1, averaged over n_batches batches, (T,)
    torch.manual_seed(0)  # the same validation chunks for every model
    total = torch.zeros(T, device=device)
    for _ in range(n_batches):
        x, y = get_text_batch("val", T)
        logits, _ = model(x)
        losses = ...  # TODO: the loss at every position of every sequence, (batch_size, T)
        total += ...  # TODO: the average over the batch, (T,)
    return (total / n_batches).cpu()

In [ ]:
# ✅ Check your loss_per_position
curve = loss_per_position(lm_models["learned"], BLOCK_SIZE, n_batches=4)
assert curve.shape == (BLOCK_SIZE,), f"expected one loss per position, got shape {tuple(curve.shape)}"
assert curve[0] > curve[-10:].mean(), "position 0 sees a single character and should have a higher loss than the last positions"
assert 1.0 < curve.mean() < 3.0, f"the average validation loss should be around 2, got {curve.mean():.2f}"
print("Looks good ✅")

print(f"\n{'':<11}{'val loss at training length':>30}")
for pos, model in lm_models.items():
    print(f"{pos:<11}{loss_per_position(model, BLOCK_SIZE).mean():>30.3f}")

**4.3 Compare the five methods at the training length.** Are the differences large? Is the result for `"none"` what you expected after 3.2?

---

*Your answer here:*

---

## 5. Length extrapolation

Training cost grows with the sequence length, so models are often trained on shorter sequences than they are later used on. What happens on sequences **four times longer** than anything seen in training? (Learned positions cannot even be evaluated: there is no embedding for position 64.)

**5.1 Run the cell below.** It plots the validation loss at each position of 256-character sequences. Left of the dashed line, the positions were seen in training.

In [ ]:
LONG = 4 * BLOCK_SIZE
long_curves = {pos: loss_per_position(model, LONG) for pos, model in lm_models.items() if pos != "learned"}

plt.figure(figsize=(9, 4))
for pos, curve in long_curves.items():
    plt.plot(smooth(curve.tolist(), window=8), label=pos)
plt.axvline(BLOCK_SIZE, color="gray", linestyle="--", label="training length")
plt.ylim(1.0, 5.0)
plt.xlabel("position")
plt.ylabel("validation loss (smoothed)")
plt.legend()
plt.show()

print(f"{'':<11}{'positions < 64':>16}{'positions 64-255':>18}")
for pos, curve in long_curves.items():
    print(f"{pos:<11}{curve[:BLOCK_SIZE].mean():>16.3f}{curve[BLOCK_SIZE:].mean():>18.3f}")

**5.2 Describe the curves.** Which methods extrapolate and which do not? Explain each result with how the method encodes position.

---

*Your answer here:*

---

### Can RoPE be rescued without retraining?

When Llama models were extended from 2,048 to longer contexts, a simple trick turned out to work: **position interpolation** ([Chen et al., 2023](https://arxiv.org/abs/2306.15595)). Instead of extrapolating to unseen angles, multiply all positions by $\frac{\text{training length}}{\text{new length}}$, so that the 256 positions are squeezed into the range of angles the model saw in training. In practice this is followed by a short fine-tuning run at the new length.

**5.3 Set `rope_scale` to `BLOCK_SIZE / LONG` in every attention layer of the RoPE model**, evaluate it on the long sequences again, and then set it back to 1.

Hint: `model.blocks` is an `nn.Sequential` of `Block`s, and each block has an `attn` with the attribute `rope_scale`.

In [ ]:
rope_model = lm_models["rope"]

...  # TODO: set rope_scale = BLOCK_SIZE / LONG in every attention layer
interpolated = loss_per_position(rope_model, LONG)
...  # TODO: set rope_scale back to 1.0

plt.figure(figsize=(9, 3))
plt.plot(smooth(long_curves["rope"].tolist(), window=8), label="rope")
plt.plot(smooth(interpolated.tolist(), window=8), label="rope, position interpolation")
plt.axvline(BLOCK_SIZE, color="gray", linestyle="--")
plt.ylim(1.0, 5.0)
plt.xlabel("position")
plt.ylabel("validation loss (smoothed)")
plt.legend()
plt.show()
print(f"positions 64-255: {long_curves['rope'][BLOCK_SIZE:].mean():.3f} without, {interpolated[BLOCK_SIZE:].mean():.3f} with interpolation")

In [ ]:
# ✅ Check that the RoPE model is back to normal
assert all(block.attn.rope_scale == 1.0 for block in rope_model.blocks), "set rope_scale back to 1.0"
assert not torch.allclose(interpolated, long_curves["rope"]), "the interpolated curve should differ: did you change rope_scale before evaluating?"
print("Looks good ✅")

**5.4 What did interpolation do to the curve, both before and after position 64?** Why would interpolation work much better after a short fine-tuning run at the new length?

---

*Your answer here:*

---

## 6. A task that needs exact positions

Language modelling is forgiving: most of the signal is local, and approximate positions are enough. Now a task where position is **everything**. The input is a random sequence of tokens from a vocabulary of 16, and at every position the model has to output the token from exactly **5 positions earlier**:

```
x   7  2  9  9  4 11  0  3  ...
y   -  -  -  -  -  7  2  9  ...      (- = no target, -100)
```

The tokens are random, so there is no content to exploit: the model has to attend to *exactly* the token at distance 5. Chance level is 1/16 ≈ 6%.

**6.1 Complete `get_lookback_batch`.**

In [ ]:
LOOKBACK = 5
LB_VOCAB = 16


def get_lookback_batch(T, batch_size=32):
    # Random token sequences x and targets y with y[:, t] = x[:, t - LOOKBACK], and -100 for the first LOOKBACK positions
    x = torch.randint(LB_VOCAB, (batch_size, T))
    y = torch.full((batch_size, T), -100)
    ...  # TODO: fill in y from position LOOKBACK onwards
    return x.to(device), y.to(device)

In [ ]:
# ✅ Check your batches
x, y = get_lookback_batch(20, batch_size=3)
assert x.shape == y.shape == (3, 20)
assert torch.all(y[:, :LOOKBACK] == -100), "the first LOOKBACK positions have no target"
assert torch.equal(y[:, LOOKBACK:], x[:, :-LOOKBACK]), "y[:, t] should be x[:, t - LOOKBACK]"
print("Looks good ✅")

**6.2 Train the five methods on sequences of length 32**, and test them on length 32 and on length 128. This takes roughly 2–4 minutes on a CPU.

In [ ]:
LB_TRAIN_LEN = 32
LB_STEPS = 1500


@torch.no_grad()
def lookback_accuracy(model, T, n_batches=10):
    # The fraction of positions (with a target) at which the model predicts the right token
    torch.manual_seed(0)
    correct = total = 0
    for _ in range(n_batches):
        x, y = get_lookback_batch(T)
        preds = model(x)[0].argmax(dim=-1)
        mask = y != -100
        correct += (preds[mask] == y[mask]).sum().item()
        total += mask.sum().item()
    return correct / total


lb_results = {}
for pos in POSITION_METHODS:
    start = time.time()
    model = make_model(pos, LB_VOCAB, LB_TRAIN_LEN, n_layer=2)
    train(model, lambda: get_lookback_batch(LB_TRAIN_LEN), LB_STEPS, lr=3e-3)
    short = lookback_accuracy(model, LB_TRAIN_LEN)
    long = lookback_accuracy(model, 4 * LB_TRAIN_LEN) if pos != "learned" else float("nan")
    lb_results[pos] = (short, long)
    print(f"{pos:<11} accuracy at length {LB_TRAIN_LEN}: {short:6.1%}   at length {4 * LB_TRAIN_LEN}: {long:6.1%}   ({time.time() - start:.0f} s)")

**6.3 Which methods solve the task, and which of them still solve it on longer sequences?** Explain the results with what each method can and cannot express. For any method that surprised you, try to come up with an explanation (or change `LB_STEPS` or `LOOKBACK` and see what happens).

---

*Your answer here:*

---

### Summary

**6.4 Fill in the table** with what *you* observed: ✓, ✗ or ~ (partly).

| | none | learned | sinusoidal | RoPE | ALiBi |
|---|---|---|---|---|---|
| language modelling at the training length | | | | | |
| language modelling beyond the training length | | | | | |
| exact relative positions (look-back task) | | | | | |
| look-back task beyond the training length | | | | | |
| extra parameters | | | | | |

**6.5 If you were to pretrain a new LLM today, which positional encoding would you choose, and why?** What would you do if users later wanted to give it inputs much longer than the training length?

---

*Your answer here:*

---

## 7. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 7.1. Absolute and relative

Which of these positional encodings are *relative*, i.e. make the attention depend on the distance between tokens rather than on their absolute positions?

A. Learned position embeddings and sinusoidal encodings<br>
B. RoPE and ALiBi<br>
C. Only sinusoidal encodings<br>
D. None of them: attention always uses absolute positions<br>

**Answer:** 

---

### 7.2. RoPE

What does RoPE do?

A. It adds a learned vector for each position to the token embeddings<br>
B. It rotates the queries and keys by angles proportional to their positions<br>
C. It adds a penalty proportional to the distance to the attention scores<br>
D. It rotates the values by angles proportional to their positions<br>

**Answer:** 

---

### 7.3. Learned position embeddings

What is the main limitation of learned absolute position embeddings?

A. They make training much slower<br>
B. They cannot be used with a causal mask<br>
C. They do not exist for positions beyond the maximum length used when the model was built<br>
D. They are only defined for encoder models<br>

**Answer:** 

---

### 7.4. ALiBi

Why does ALiBi extrapolate well to sequences longer than those seen in training?

A. Its bias depends only on the distance and penalises far-away tokens so much that, locally, a long sequence looks like a short one<br>
B. It learns a separate embedding for every possible length<br>
C. It removes the causal mask for long sequences<br>
D. It uses position interpolation automatically<br>

**Answer:** 

---

### 7.5. No positional encoding

A *decoder* without any positional encoding (NoPE) can still learn something about positions. Where does that information come from?

A. From the layer norm<br>
B. From the causal mask, because position $t$ can only attend to $t + 1$ tokens<br>
C. From the tokenizer, which adds position numbers to the tokens<br>
D. It cannot: a decoder without positional encoding is a bag-of-words model<br>

**Answer:** 